# Exploring ACL2 with MCP Tools

This notebook demonstrates how to use two MCP servers together:

1. **acl2-kg-mcp** — A Knowledge Graph of ~400K ACL2 symbols, ~500K code cells, and 14K notebooks, stored in Weaviate
2. **acl2-mcp** — A live ACL2 theorem prover session

The workflow: **search** the KG → **understand** the code → **run & prove** it in ACL2.

We'll use Python to call the MCP tools directly via their underlying modules.

In [1]:
# Setup: import the KG client and ACL2 runner
from acl2_kg_mcp import weaviate_client as kg
from acl2_mcp.server import call_tool as acl2_call
import json

def show(data):
    """Pretty-print a dict."""
    print(json.dumps(data, indent=2, default=str))

2026-03-01 02:07:46,275 - mcp.server.lowlevel.server - DEBUG - Initializing server 'acl2-mcp'
2026-03-01 02:07:46,276 - mcp.server.lowlevel.server - DEBUG - Registering handler for ListToolsRequest
2026-03-01 02:07:46,276 - mcp.server.lowlevel.server - DEBUG - Registering handler for CallToolRequest


## 1. The Knowledge Graph at a Glance

The ACL2 KG contains the entire ACL2 community books — every function, macro, theorem, and constant — indexed and searchable.

In [2]:
stats = kg.get_stats()

print("=== ACL2 Knowledge Graph ===")
for name, count in stats["collections"].items():
    print(f"  {name:20s} {count:>10,}")
print()
print("Symbol kinds:")
for kind, count in sorted(stats["symbol_kinds"].items(), key=lambda x: -x[1]):
    print(f"  {kind:20s} {count:>10,}")

2026-03-01 02:09:32,398 - httpcore.connection - DEBUG - connect_tcp.started host='host.docker.internal' port=8080 local_address=None timeout=5.0 socket_options=None
2026-03-01 02:09:32,400 - httpcore.connection - DEBUG - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0xffff7a64e270>
2026-03-01 02:09:32,400 - httpcore.http11 - DEBUG - send_request_headers.started request=<Request [b'GET']>
2026-03-01 02:09:32,401 - httpcore.http11 - DEBUG - send_request_headers.complete
2026-03-01 02:09:32,401 - httpcore.http11 - DEBUG - send_request_body.started request=<Request [b'GET']>
2026-03-01 02:09:32,401 - httpcore.http11 - DEBUG - send_request_body.complete
2026-03-01 02:09:32,401 - httpcore.http11 - DEBUG - receive_response_headers.started request=<Request [b'GET']>
2026-03-01 02:09:32,402 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 404, b'Not Found', [(b'Access-Control-Allow-Headers', b'Content-Type, Authorization,

=== ACL2 Knowledge Graph ===
  ACL2Notebook             14,544
  ACL2Cell                507,645
  ACL2Symbol              403,038
  ACL2Summary              15,159
  DoclingPapers            14,094

Symbol kinds:
  theorem                 146,967
  unknown                 143,156
  function                 85,716
  macro                    19,498
  constant                  5,721
  raw-function              1,272
  stobj                       362
  variable                    326
  special-form                 20


## 2. Semantic Symbol Search

The KG stores vector embeddings for every symbol (and every code cell). We can search
by *meaning* rather than by name. When direct symbol-vector matches are weak, the
search automatically falls back to code-cell embeddings and harvests the symbols
defined there — giving much better results for conceptual queries.

In [4]:
results = kg.search_symbols("binary search tree insertion and lookup", limit=5)

for sym in results["results"]:
    src = f"  (via {sym['source']})" if "source" in sym else ""
    print(f"{sym['distance']:.3f}  {sym['kind']:12s}  {sym['qualified_name']}{src}")
    print(f"       └─ {sym.get('source_file', 'N/A')}")

0.326  theorem       TREESET::TREE-SEARCH-IN-BECOMES-TREE-IN-WHEN  (via code_cell)
       └─ N/A
0.342  function      TREESET::TREE-INSERT  (via code_cell)
       └─ N/A
0.347  theorem       TREESET::TREE-INSERT.INP  (via code_cell)
       └─ N/A
0.348  theorem       ACL2::MATCH-TREE-RESTRICTIONS-OF-LOOKUP  (via code_cell)
       └─ N/A
0.348  theorem       TREESET::TREE-IN-WHEN-TREE-SEARCH-IN  (via code_cell)
       └─ N/A


## 3. Exploring a Symbol

Let's drill into a specific symbol to see its definition, dependencies, and dependents.
We'll look at `ACL2::COMPARABLE-MERGESORT` — a verified merge-sort implementation.

In [6]:
sym = kg.get_symbol("ACL2::COMPARABLE-MERGESORT")

print(f"Name:    {sym['qualified_name']}")
print(f"Kind:    {sym['kind']}")
print(f"Package: {sym['package']}")

defn = sym.get("definition", {})
print(f"File:    {defn.get('notebook_source', 'N/A')}")
print(f"\nDefinition (code):\n{defn.get('code', '')[:500]}")

print(f"\nDependencies ({sym['dependencies']['total']}):")
for d in sym['dependencies']['results'][:5]:
    print(f"  → {d['qualified_name']}  ({d['kind']})")

print(f"\nDependents ({sym['dependents']['total']}):")
for d in sym['dependents']['results'][:5]:
    print(f"  ← {d['qualified_name']}  ({d['kind']})")

Name:    ACL2::COMPARABLE-MERGESORT
Kind:    function
Package: ACL2
File:    books/defsort/generic.lisp

Definition (code):
(defund comparable-mergesort (x)
  (declare (xargs :measure (len x)
                  :guard (comparable-listp x)
                  :verify-guards nil))
  (mbe :logic (cond ((atom x)
                     nil)
                    ((atom (cdr x))
                     (list (car x)))
                    (t
                     (let ((half (floor (len x) 2)))
                       (comparable-merge
                        (comparable-mergesort (take half x))
                        (comparable-merg

Dependencies (23):
  → ACL2::COMPARABLE-LISTP  (function)
  → ACL2::COMPARABLE-MERGE  (function)
  → ACL2::DEFUND  (macro)
  → ACL2::FAST-COMPARABLE-MERGESORT-FIXNUMS  (function)
  → ACL2::FAST-COMPARABLE-MERGESORT-INTEGERS  (function)

Dependents (33):
  ← ACL2::APPEND-COMPARE-EXTRACTS-OF-TAKE/NTHCDR  (theorem)
  ← ACL2::COMMON-<<-SORT-FOR-PERMS  (theorem)
  ← ACL2::COMM

## 4. From the KG to a Live ACL2 Session

The `get_include_book` tool bridges the KG and the prover: given a source file
from the KG, it produces the `(include-book ...)` form needed to load that book
in a live ACL2 session — including any portcullis (prerequisite) commands.

In [8]:
ib = kg.get_include_book("books/defsort/generic.lisp")

print(f"Source file:  {ib['source_file']}")
print(f"Include form: {ib['include_book']}")
if ib.get("portcullis"):
    print(f"\nPortcullis (pre-req commands):")
    for cmd in ib["portcullis"][:3]:
        print(f"  {str(cmd)[:120]}")
else:
    print("\nNo portcullis needed — this book can be included directly.")

Source file:  books/defsort/generic.lisp
Include form: (include-book "defsort/generic" :dir :system)

Portcullis (pre-req commands):
  {'filename': 'cert.acl2', 'content': '(include-book "std/portcullis" :dir :system)\n(include-book "data-structures/portc


## 5. Live ACL2 Session

Now we use the **acl2-mcp** server to start a live ACL2 prover session, define
functions, prove theorems, and execute code — all from Python.

We'll define a simple list-reversal function `my-rev`, prove a theorem about it,
and test it.

In [9]:
# Helper to call acl2-mcp tools (they're async)
async def acl2(tool, **kwargs):
    """Call an acl2-mcp tool and return the text result."""
    result = await acl2_call(tool, kwargs)
    return result[0].text

# Start a fresh ACL2 session
output = await acl2("start_session", name="demo")
print(output)

2026-03-01 02:12:15,036 - acl2_mcp.server - DEBUG - Starting ACL2 session with ACL2_SYSTEM_BOOKS=/home/acl2/books
2026-03-01 02:12:15,038 - acl2_mcp.server - INFO - Session d6cf1a00-2436-4eb6-88f9-ec6914e6997b created (banner will be consumed on first command)


Session started successfully. ID: d6cf1a00-2436-4eb6-88f9-ec6914e6997b


In [10]:
# Extract session ID for subsequent calls
sid = output.split("ID: ")[1].strip()

# Define a recursive list-append function
result = await acl2("admit", session_id=sid, code="""
(defun my-app (x y)
  (if (endp x)
      y
      (cons (car x)
            (my-app (cdr x) y))))
""")
print(result[:500])

2026-03-01 02:12:25,001 - acl2_mcp.server - INFO - Session d6cf1a00-2436-4eb6-88f9-ec6914e6997b: Consuming banner with marker ___BANNER_7e9bc943f0...
2026-03-01 02:12:25,003 - acl2_mcp.server - INFO - Session d6cf1a00-2436-4eb6-88f9-ec6914e6997b: Banner consumed after 26 lines


Admit succeeded:

NIL
ACL2 !>
The admission of MY-APP is trivial, using the relation O< (which is
known to be well-founded on the domain recognized by O-P) and the measure
(ACL2-COUNT X).  We observe that the type of MY-APP is described by
the theorem (OR (CONSP (MY-APP X Y)) (EQUAL (MY-APP X Y) Y)).  We used
primitive type reasoning.

Summary
Form:  ( DEFUN MY-APP ...)
Rules: ((:FAKE-RUNE-FOR-TYPE-SET NIL))
Time:  0.00 seconds (prove: 0.00, print: 0.00, other: 0.00)
 MY-APP
ACL2 !>


In [11]:
# Prove: appending NIL is identity
result = await acl2("admit", session_id=sid, code="""
(defthm my-app-nil
  (equal (my-app x nil) (true-list-fix x)))
""")
print(result[:800])

Admit succeeded:

NIL
ACL2 !>
*1 (the initial Goal, a key checkpoint) is pushed for proof by induction.

Perhaps we can prove *1 by induction.  Two induction schemes are suggested
by this conjecture.  These merge into one derived induction scheme.

We will induct according to a scheme suggested by (TRUE-LIST-FIX X),
while accommodating (MY-APP X NIL).

These suggestions were produced using the :induction rules MY-APP and
TRUE-LIST-FIX.  If we let (:P X) denote *1 above then the induction
scheme we'll use is
(AND (IMPLIES (NOT (CONSP X)) (:P X))
     (IMPLIES (AND (CONSP X) (:P (CDR X)))
              (:P X))).
This induction is justified by the same argument used to admit TRUE-LIST-FIX.
When applied to the goal at hand the above induction scheme produces
two nontautological subgoals.
Subgo


In [12]:
# Test: evaluate my-app on concrete lists
result = await acl2("evaluate", session_id=sid, code="""
(my-app '(1 2 3) '(4 5 6))
""")
print(result)

# Prove associativity of my-app
result = await acl2("admit", session_id=sid, code="""
(defthm my-app-assoc
  (equal (my-app (my-app x y) z)
         (my-app x (my-app y z))))
""")
print(result[:600])

NIL
ACL2 !>(1 2 3 4 5 6)
ACL2 !>
Admit succeeded:

NIL
ACL2 !>
*1 (the initial Goal, a key checkpoint) is pushed for proof by induction.

Perhaps we can prove *1 by induction.  Three induction schemes are
suggested by this conjecture.  Subsumption reduces that number to two.
However, one of these is flawed and so we are left with one viable
candidate.  

We will induct according to a scheme suggested by (MY-APP X Y), while
accommodating (MY-APP X (MY-APP Y Z)).

These suggestions were produced using the :induction rule MY-APP. 
If we let (:P X Y Z) denote *1 above then the induction scheme we'll
use is
(AND (IMPLIES (AND (NOT


In [13]:
# Clean up the session
result = await acl2("end_session", session_id=sid)
print(result)

Session d6cf1a00-2436-4eb6-88f9-ec6914e6997b ended successfully


---

## Summary

This demo showed the two MCP servers working together:

| Step | Tool | What it does |
|------|------|-------------|
| **Explore** | `kg_stats` | Overview of the 400K-symbol Knowledge Graph |
| **Search** | `kg_search` (symbols) | Semantic search with code-cell enrichment |
| **Inspect** | `kg_get_symbol` | Drill into definitions, dependencies, dependents |
| **Bridge** | `kg_get_include_book` | Map KG source files → ACL2 `include-book` forms |
| **Define** | `admit` | Define functions in a live ACL2 session |
| **Prove** | `admit` (defthm) | Prove theorems by induction |
| **Execute** | `evaluate` | Run ACL2 expressions and get results |

The Knowledge Graph gives you **indexed, searchable access** to the entire ACL2
community books library, and the live session lets you **interactively define,
prove, and test** new code — all from Python.